In [3]:
import pandas as pd
import joblib
import sklearn


from sklearn.metrics import roc_auc_score
from lib.earlystopping import EarlyStopping
from lib.losses import BinaryCrossEntropy
from lib.activations import Relu, Sigmoid
from lib.layers import DenseLayer, Dropout
from lib.optimizers import Adam
from lib.nnet import NNet
from lib.class_weight import compute_class_weight
import numpy as np
from sklearn.metrics import  roc_auc_score, accuracy_score

# import lightgbm as lgb

sklearn.set_config(transform_output="pandas")
pd.set_option('display.max_colwidth', None)


preprocessor = joblib.load("transformer.pkl")

df = pd.read_csv("data/balanced_train_data.csv")
val_df = pd.read_csv("data/split_data_val.csv")

y  = df['TARGET'].copy().astype(np.float32)
df = df.drop(columns=['TARGET'])
X_transformed = preprocessor.transform(df).astype(np.float32)

y_val = val_df['TARGET'].copy().astype(np.float32)
val_df = val_df.drop(columns=['TARGET'])
X_val_transformed = preprocessor.transform(val_df).astype(np.float32)

del df, val_df

X_train = X_transformed
X_val   = X_val_transformed

model = NNet(
    layers=[
        DenseLayer(input_size=X_transformed.shape[1], output_size=64, activation=Relu()),
        Dropout(0.3),
        DenseLayer(output_size=32, activation=Relu()),
        Dropout(0.2),
        DenseLayer(output_size=16, activation=Relu()),
        # Dropout(0.2),
        DenseLayer(output_size=1, activation=Sigmoid(), weights_initializer='xavier')
    ],
    optimizer=Adam(learning_rate=0.001),
    loss=BinaryCrossEntropy(),
    earlystopping=EarlyStopping(patience=40, restore_best_weights=True)
)


class_weight = compute_class_weight(np.unique(y), y)
result = model.fit(
    X_transformed,
    y,
    validation_data=(X_val_transformed, y_val),
    epochs=300,
    class_weight=class_weight,
    batch_size=256
)

del X_transformed, y
test_split_df = pd.read_csv('data/split_data_test.csv')

y_test        = test_split_df['TARGET'].astype(np.float32)
test_split_df = test_split_df.drop(columns=['TARGET'])
X_test_split  = preprocessor.transform(test_split_df).astype(np.float32)

y_prob    = model.predict(X_test_split)
y_pred    = (y_prob > 0.5).astype(int)

result = pd.DataFrame({
    'library': ["scikit-learn, tensorflow"],
    'algorithm': ['Multilayer-perceptron'],
    'hyerparamters': ['hidden layers : (64, 32, 16) + relu activation + dropout 0.3, 0.2 + early stopping learning rate 0.001 batch_size 256'],
    'accuracy': [accuracy_score(y_test, y_pred)],
    'auc': [roc_auc_score(y_test, y_prob)]
}, index=None)

print(result.T)

del test_split_df, X_test_split, y_test, y_pred

df = pd.read_csv("data/bank_data_test.csv")

ids = df['ID'].values.tolist()

X_test = preprocessor.transform(df).astype(np.float32)

y_pred = model.predict(X_test)

preds_df = pd.DataFrame(data={'ID':ids, 'TARGET':y_pred.ravel().tolist()}, columns=['ID', 'TARGET'])

preds_df.to_csv("prediction_file.csv", index=False)
print("prediction_file.csv saved")


Epoch 1/300
val loss : 12.57317851003173 Validation loss decreased. Resetting counter.


100%|===============================================================================================



Epoch 2/300
val loss : 0.27227543131983883 Validation loss decreased. Resetting counter.


100%|===============================================================================================



Epoch 3/300
val loss : 0.2506516668278502 Validation loss decreased. Resetting counter.


...


Epoch 200/300
val loss : 0.2251337762727077 EarlyStopping counter: 38 out of 40


100%|===============================================================================================



Epoch 201/300
val loss : 0.2258921354724795 EarlyStopping counter: 39 out of 40


100%|===============================================================================================



Epoch 202/300
val loss : 0.22513811339111006 EarlyStopping counter: 40 out of 40
                                                                                                                                   0
library                                                                                                     scikit-learn, tensorflow
algorithm                                                                                                      Multilayer-perceptron
hyerparamters  hidden layers : (64, 32, 16) + relu activation + dropout 0.3, 0.2 + early stopping learning rate 0.001 batch_size 256
accuracy                                                                                                                    0.919395
auc                                                                                                                         0.838542
prediction_file.csv saved
